# Starter — Steer LLMs to Yap About the Ocean

> ⚠️ **This notebook requires a GPU** for comfortable LLM inference (Kaggle / Colab).
> It is a documented starter, not an executed baseline.

**Competition:** make a language model **steer every answer toward a trait** — here,
*the ocean* — while still genuinely answering the user's question. For each prompt in
`prompts.csv` you produce the model's answer; submissions are scored on:

- `trait_score` — how much the answer talks about the ocean (higher = better)
- `quality_score` — how well it still answers the original question (higher = better)

The trick is the trade-off: replying "THE OCEAN IS GREAT" to everything maxes the
trait but zeroes the quality.

- **Kaggle link:** _TODO: add link_

In [ ]:
# pip install transformers accelerate
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_DIR = "."
prompts = pd.read_csv(f"{DATA_DIR}/prompts.csv")
prompts.head()

## Baseline idea 1 — system-prompt steering (no training)

The simplest baseline: tell the model to weave the ocean into every answer.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"      # small instruct model
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto",
                                             device_map="auto")

SYSTEM = ("You love the ocean deeply. Whatever you are asked, answer helpfully and "
          "completely, but always connect your answer to the ocean, the sea, marine "
          "life, or coastal imagery in a natural way.")

def answer(user_prompt, max_new_tokens=200):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

rows = [{"row_id": r.row_id, "answer": answer(r.prompt)} for r in prompts.itertuples()]
pd.DataFrame(rows).to_csv("answers.csv", index=False)

## Baseline idea 2 — activation steering (the IOAI-flavored approach)

Instead of prompting, add a **steering vector** to the residual stream at generation
time:

1. Run the model on contrast pairs ("Tell me about the ocean" vs neutral text) and
   record hidden states at a middle layer.
2. The **difference of means** is the "ocean direction".
3. During generation, hook that layer and add `alpha * direction` to the activations.
4. Tune `alpha`: too low = no ocean, too high = incoherent yapping (quality collapses).

This is exactly the *representation engineering / activation addition* technique — it
keeps answers on-topic better than prompting at the same trait strength, and it's a
great IOAI talking point.

## Ideas to improve

- Grid-search `alpha` and the hooked layer against the two scores.
- Combine both: mild system prompt + mild steering vector.
- Post-process: if an answer never mentions the sea, regenerate it.
